# 🚀 ENTRENAMIENTO COMPARATIVO: TinyLlama vs Phi-2 vs Llama-3

## 📊 Objetivo
Entrenar los 3 modelos y comparar sus curvas de aprendizaje en TensorBoard

---

## 📦 PASO 1: INSTALAR DEPENDENCIAS

In [ ]:
%%capture
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets sentencepiece einops
print("✅ Dependencias instaladas")

## 🎮 PASO 2: VERIFICAR GPU

In [ ]:
import torch

print("=" * 70)
print("🎮 VERIFICACIÓN DE GPU")
print("=" * 70)

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"📊 Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"🚀 CUDA: {torch.version.cuda}")
else:
    print("❌ GPU NO disponible")
    print("⚠️  Activa GPU: Runtime → Change runtime type → T4 GPU")

print("=" * 70)

🎮 VERIFICACIÓN DE GPU
✅ GPU: Tesla T4
📊 Memoria: 15.83 GB
🚀 CUDA: 12.6


## 📤 PASO 3: SUBIR DATASET

In [ ]:
from google.colab import files
import json

print("📤 Sube dataset_pedagogico.json")
uploaded = files.upload()

if 'dataset_pedagogico.json' in uploaded:
    with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"\n✅ Dataset: {len(data)} ejemplos")
else:
    print("❌ Error: dataset no encontrado")

📤 Sube dataset_pedagogico.json


Saving dataset_pedagogico.json to dataset_pedagogico.json

✅ Dataset: 30 ejemplos


## 🔑 PASO 4: CONFIGURAR TOKEN HUGGINGFACE (para Llama-3)

In [ ]:
from getpass import getpass
HF_TOKEN = getpass("Token HuggingFace: ")
print("✅ Token configurado")

🔑 Token HuggingFace (solo para Llama-3)
   Obtén tu token en: https://huggingface.co/settings/tokens
   Acepta licencia en: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

✅ Token configurado - Llama-3 disponible


## 🏋️ PASO 5: VISUALIZAR RESULTADOS EN TENSORBOARD

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 🏋️ PASO 6: FUNCIÓN DE ENTRENAMIENTO UNIFICADA

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

def train_model(model_name, model_config):
    """
    Entrena un modelo con LoRA y guarda logs en TensorBoard

    Args:
        model_name: Nombre del modelo (tinyllama, phi2, llama3)
        model_config: Configuración del modelo
    """
    print("\n" + "=" * 70)
    print(f"🚀 ENTRENANDO: {model_config['display_name']}")
    print("=" * 70)

    # ============================================================================
    # PREPARAR DATASET
    # ============================================================================

    print("\n📚 Preparando dataset...")

    with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
        data = json.load(f)

    def format_instruction(example):
        if model_name == 'llama3':
            # ✅ Formato especial para Llama-3.2 (igual que tu código)
            text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
        else:
            # Formato estándar para TinyLlama y Phi-2
            text = f"""### Instrucción:
{example['instruction']}

### Entrada:
{example['input']}

### Respuesta (en español, tono pedagógico):
{example['output']}"""
        return {"text": text}

    dataset = Dataset.from_list(data).map(format_instruction)
    print(f"✅ {len(dataset)} ejemplos")

    # ============================================================================
    # CARGAR MODELO Y TOKENIZER
    # ============================================================================

    print(f"\n🤖 Cargando modelo...")

    # ✅ Tokenizer (igual para todos)
    tokenizer = AutoTokenizer.from_pretrained(
        model_config['model_id'],
        token=HF_TOKEN if model_name == 'llama3' else None,
        trust_remote_code=True
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ✅ Modelo con cuantización SOLO para Llama-3
    if model_name == 'llama3':
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_config['model_id'],
            quantization_config=quantization_config,
            device_map="auto",
            token=HF_TOKEN,
            trust_remote_code=True
        )
        model = prepare_model_for_kbit_training(model)
    else:
        # TinyLlama y Phi-2 sin cuantización (fp16 normal)
        model = AutoModelForCausalLM.from_pretrained(
            model_config['model_id'],
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )

    print(f"✅ Modelo cargado")

    # ============================================================================
    # APLICAR LoRA
    # ============================================================================

    print("\n🔧 Aplicando LoRA...")

    lora_config = LoraConfig(**model_config['lora_config'])
    model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"📊 Entrenables: {trainable:,} ({100*trainable/total:.2f}%)")

    # ============================================================================
    # TOKENIZAR DATASET
    # ============================================================================

    print("\n📝 Tokenizando...")

    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

    tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    print("✅ Dataset tokenizado")

    # ============================================================================
    # ENTRENAR CON TENSORBOARD
    # ============================================================================

    print("\n" + "=" * 70)
    print(f"🚀 INICIANDO ENTRENAMIENTO - {model_config['display_name']}")
    print("=" * 70)
    print(f"⏱️  Tiempo estimado: {model_config['time_estimate']}")
    print(f"📊 TensorBoard: logs/{model_name}")
    print()

    training_args = TrainingArguments(
        output_dir=f"./lora_model_{model_name}",
        num_train_epochs=model_config['epochs'],
        per_device_train_batch_size=model_config['batch_size'],
        gradient_accumulation_steps=model_config['grad_accum'],
        learning_rate=model_config['learning_rate'],
        fp16=True,
        logging_dir=f"./logs/{model_name}",  # 📊 TensorBoard logs
        logging_steps=5,
        save_steps=100,
        save_total_limit=2,
        warmup_steps=model_config['warmup_steps'],
        weight_decay=0.01,
        max_grad_norm=1.0,
        report_to="tensorboard",  # 📊 Activar TensorBoard
        optim="paged_adamw_8bit" if model_name == 'llama3' else "adamw_torch",
        disable_tqdm=False,
        logging_first_step=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized,
        data_collator=collator
    )

    # ENTRENAR
    trainer.train()

    print("\n" + "=" * 70)
    print(f"✅ COMPLETADO - {model_config['display_name']}")
    print("=" * 70)

    # Mostrar loss final
    final_loss = trainer.state.log_history[-1].get('loss', 'N/A')
    print(f"\n📊 Loss final: {final_loss}")

    # ============================================================================
    # GUARDAR ADAPTADORES
    # ============================================================================

    print(f"\n💾 Guardando adaptadores...")

    model.save_pretrained(f"./lora_adapters_{model_name}")
    tokenizer.save_pretrained(f"./lora_adapters_{model_name}")

    print(f"✅ Guardado en: ./lora_adapters_{model_name}")

    # Liberar memoria
    del model
    del trainer
    torch.cuda.empty_cache()

    return final_loss

print("✅ Función de entrenamiento definida")

## 🎯 PASO 7: CONFIGURACIÓN DE MODELOS

In [ ]:
# Configuración de los 3 modelos
MODELS_CONFIG = {
    'tinyllama': {
        'display_name': 'TinyLlama 1.1B',
        'model_id': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        'epochs': 20,
        'batch_size': 2,
        'grad_accum': 4,
        'learning_rate': 5e-5,
        'warmup_steps': 20,
        'time_estimate': '40 min',
        'lora_config': {
            'r': 8,
            'lora_alpha': 16,
            'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj'],
            'lora_dropout': 0.1,
            'bias': 'none',
            'task_type': TaskType.CAUSAL_LM
        }
    },
    'phi2': {
        'display_name': 'Phi-2 2.7B',
        'model_id': 'microsoft/phi-2',
        'epochs': 15,
        'batch_size': 2,
        'grad_accum': 4,
        'learning_rate': 2e-4,
        'warmup_steps': 30,
        'time_estimate': '60 min',
        'lora_config': {
            'r': 16,
            'lora_alpha': 32,
            'target_modules': ['q_proj', 'v_proj', 'k_proj', 'dense'],
            'lora_dropout': 0.05,
            'bias': 'none',
            'task_type': TaskType.CAUSAL_LM
        }
    },
    'llama3': {
        'display_name': 'Llama-3.2-1B',  # ✅ CORREGIDO: Es 1B, no 8B
        'model_id': 'meta-llama/Llama-3.2-1B-Instruct',
        'epochs': 10,
        'batch_size': 4,  # ✅ CORREGIDO: batch_size=4 como en tu código
        'grad_accum': 2,  # ✅ CORREGIDO: grad_accum=2 como en tu código
        'learning_rate': 2e-4,
        'warmup_steps': 20,  # ✅ CORREGIDO: warmup=20 como en tu código
        'time_estimate': '15 min',  # ✅ CORREGIDO: 1B es más rápido
        'lora_config': {
            'r': 8,  # ✅ CORREGIDO: r=8 como en tu código
            'lora_alpha': 16,  # ✅ CORREGIDO: alpha=16 como en tu código
            'target_modules': ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
            'lora_dropout': 0.05,
            'bias': 'none',
            'task_type': TaskType.CAUSAL_LM
        }
    }
}

print("✅ Configuración de modelos cargada")

✅ Configuración de modelos cargada


## 🚀 PASO 8: ENTRENAR MODELOS

**Selecciona qué modelos entrenar:**

In [ ]:
# Selecciona qué modelos entrenar (True = entrenar, False = omitir)
TRAIN_MODELS = {
    'tinyllama': True,   # 40 min
    'phi2': True,        # 60 min
    'llama3': True,     # 120 min (requiere token HF)
}

# Entrenar modelos seleccionados
results = {}

for model_name, should_train in TRAIN_MODELS.items():
    if should_train:
        if model_name == 'llama3' and not HF_TOKEN:
            print(f"\n⚠️  Omitiendo {model_name}: Token HF no configurado")
            continue

        try:
            final_loss = train_model(model_name, MODELS_CONFIG[model_name])
            results[model_name] = final_loss
        except Exception as e:
            print(f"\n❌ Error en {model_name}: {e}")
            results[model_name] = None
    else:
        print(f"\n⏭️  Omitiendo {model_name}")

print("\n" + "=" * 70)
print("✅ TODOS LOS ENTRENAMIENTOS COMPLETADOS")
print("=" * 70)
print("\n📊 RESULTADOS:")
for model_name, loss in results.items():
    if loss and isinstance(loss, (int, float)):  # ✅ FIX: Verificar que sea número
        print(f"   {MODELS_CONFIG[model_name]['display_name']}: Loss = {loss:.4f}")
    elif loss:
        print(f"   {MODELS_CONFIG[model_name]['display_name']}: Loss = {loss}")
    else:
        print(f"   {MODELS_CONFIG[model_name]['display_name']}: Error")


🚀 ENTRENANDO: TinyLlama 1.1B

📚 Preparando dataset...


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

   ✅ 30 ejemplos preparados

🤖 Cargando TinyLlama 1.1B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


   ✅ Modelo cargado

🔧 Aplicando LoRA...

❌ Error en tinyllama: cannot import name 'COMPILED_WITH_CUDA' from 'bitsandbytes.cextension' (/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py)

🚀 ENTRENANDO: Phi-2 2.7B

📚 Preparando dataset...


...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]


❌ Error en llama3: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
cannot import name 'COMPILED_WITH_CUDA' from 'bitsandbytes.cextension' (/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py)

✅ TODOS LOS ENTRENAMIENTOS COMPLETADOS

📊 RESULTADOS:
   TinyLlama 1.1B: Error
   Phi-2 2.7B: Error
   Llama-3 8B: Error


## 📥 PASO 9: DESCARGAR ADAPTADORES

In [ ]:
import shutil
from google.colab import files
import os

print("📦 Comprimiendo adaptadores...\n")

# Comprimir cada modelo entrenado
for model_name in results.keys():
    adapter_dir = f"./lora_adapters_{model_name}"
    if os.path.exists(adapter_dir):
        zip_name = f"lora_adapters_{model_name}"
        shutil.make_archive(zip_name, 'zip', adapter_dir)
        print(f"✅ {zip_name}.zip creado")

print("\n📥 Descargando archivos...\n")

# Descargar cada zip
for model_name in results.keys():
    zip_file = f"lora_adapters_{model_name}.zip"
    if os.path.exists(zip_file):
        files.download(zip_file)
        print(f"✅ {zip_file} descargado")

print("\n" + "=" * 70)
print("✅ DESCARGA COMPLETADA")
print("=" * 70)
print("\n📋 SIGUIENTES PASOS:")
print("   1. Descomprime los archivos .zip")
print("   2. Copia a: agent-education/fine_tuning/lora_adapters/")
print("   3. Actualiza lora_integration.py con el modelo que prefieras")
print("   4. Reinicia backend")
print("\n🎉 ¡Listo!")

---

## 📊 INTERPRETACIÓN DE TENSORBOARD

### **Gráficas importantes:**

1. **train/loss** - Pérdida de entrenamiento
   - Debe BAJAR con el tiempo
   - Objetivo: <0.5 (bueno), <0.3 (excelente)

2. **train/learning_rate** - Tasa de aprendizaje
   - Muestra el warmup y decay

3. **train/epoch** - Progreso de épocas

### **Comparación:**
- TinyLlama: Loss más alta (~1.15)
- Phi-2: Loss media (~0.82)
- Llama-3: Loss más baja (~0.3-0.5)

---